In [ ]:
# =========================
# 0. Install / Imports
# =========================
!pip install joblib scikit-learn pandas -q

import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
# =========================
# 1. Simulate Larger Dataset - Make delivery_failed and breakdown_risk depend on realistic conditions.
# =========================
import numpy as np

np.random.seed(42)

rows = []

for _ in range(100):

    distance_km = np.random.randint(5, 25)
    num_stops = np.random.randint(1, 10)
    hub_congestion = np.random.rand()
    traffic_level = np.random.rand()
    weather_rain = np.random.choice([0, 1])
    vehicle_mileage_km = np.random.randint(20000, 250000)
    days_since_service = np.random.randint(10, 200)
    driver_experience = np.random.randint(1, 10)
    current_load = np.random.randint(10, 100)

    parcel_volume = int(
        50
        + distance_km * 3
        + num_stops * 5
        + hub_congestion * 60
        + traffic_level * 30
        + weather_rain * 20
        + np.random.normal(0, 10)
    )

    parcel_volume = max(30, min(parcel_volume, 300))

    delivery_failed = int(
        traffic_level > 0.7
        or hub_congestion > 0.75
        or (weather_rain == 1 and current_load > 70)
    )

    breakdown_risk = int(
        (
            vehicle_mileage_km > 160000
            or days_since_service > 120
        )
        and np.random.rand() > 0.1  # 10% noise
    )

    rows.append({
        "distance_km": distance_km,
        "num_stops": num_stops,
        "hub_congestion": hub_congestion,
        "traffic_level": traffic_level,
        "weather_rain": weather_rain,
        "vehicle_mileage_km": vehicle_mileage_km,
        "days_since_service": days_since_service,
        "driver_experience": driver_experience,
        "current_load": current_load,
        "parcel_volume": parcel_volume,
        "delivery_failed": delivery_failed,
        "breakdown_risk": breakdown_risk,
    })

data = pd.DataFrame(rows)

data.head()

data

In [ ]:
# =========================
# 2. Define Features
# =========================
FEATURES = [
    "distance_km",
    "num_stops",
    "hub_congestion",
    "traffic_level",
    "weather_rain",
    "vehicle_mileage_km",
    "days_since_service",
    "driver_experience",
    "current_load",
]

X = data[FEATURES]

y_demand = data["parcel_volume"]          # regression
y_failure = data["delivery_failed"]       # classification
y_maintenance = data["breakdown_risk"]    # classification

In [ ]:
# =========================
# 3. Train Thin-Slice Models
# =========================

demand_model = RandomForestRegressor(
    n_estimators=20,
    random_state=42
)

failure_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

maintenance_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

demand_model.fit(X, y_demand)
failure_model.fit(X, y_failure)
maintenance_model.fit(X, y_maintenance)

print("✅ Models trained.")

Regression: MAE / RMSE


Classification: accuracy / classification report

In [ ]:
# =========================
# 3A. Model Evaluation
# =========================
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    classification_report,
)
import numpy as np

# Predictions on training data
# Note: This is training-set evaluation only because this is a thin-slice prototype.
demand_pred = demand_model.predict(X)
failure_pred = failure_model.predict(X)
maintenance_pred = maintenance_model.predict(X)

# Regression evaluation
demand_mae = mean_absolute_error(y_demand, demand_pred)
demand_rmse = np.sqrt(mean_squared_error(y_demand, demand_pred))

print("========== Demand Forecasting Model ==========")
print(f"MAE : {demand_mae:.2f}")
print(f"RMSE: {demand_rmse:.2f}")

# Delivery failure classification evaluation
failure_accuracy = accuracy_score(y_failure, failure_pred)

print("\n========== Delivery Failure Model ==========")
print(f"Accuracy: {failure_accuracy:.3f}")
print(classification_report(y_failure, failure_pred))

# Maintenance classification evaluation
maintenance_accuracy = accuracy_score(y_maintenance, maintenance_pred)

print("\n========== Predictive Maintenance Model ==========")
print(f"Accuracy: {maintenance_accuracy:.3f}")
print(classification_report(y_maintenance, maintenance_pred))

These metrics are calculated on simulated training data for prototype validation only.


A production version should use train/test split, cross-validation, and real historical operational data.

In [ ]:
# =========================
# 4. Export Models + Feature List
# =========================
joblib.dump(demand_model, "demand_model.joblib")
joblib.dump(failure_model, "delivery_failure_model.joblib")
joblib.dump(maintenance_model, "maintenance_model.joblib")
joblib.dump(FEATURES, "features.joblib")

print("✅ Saved:")
print("- demand_model.joblib")
print("- delivery_failure_model.joblib")
print("- maintenance_model.joblib")
print("- features.joblib")